In [ ]:
import pandas as pd
from harbor.analysis.cross_docking import DockingDataModel, PoseSelector

In [ ]:
posedf = pd.read_csv("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/docked_ligand_rmsds/rmsd_results.csv")

In [ ]:
posit_data = DockingDataModel.deserialize("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/ALL_combined_results.parquet")

In [ ]:
posit_data.dataframe.groupby(["Query_Ligand", "Reference_Ligand"]).nunique()

In [ ]:
df = posit_data.dataframe
df[(df["Query_Ligand"] == "AAR-POS-0daf6b7e-1")&(df["Reference_Ligand"] == "Mpro-P0025_0A")]

In [ ]:
df[(df["Query_Ligand"] == "AAR-POS-0daf6b7e-1")&(df["Reference_Ligand"] == "AAR-POS-d2a4d1df-21")]

In [ ]:
tcdf = pd.read_csv("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/tanimoto_combo/tanimoto_combo.csv")

In [ ]:
tcdf.nunique()

In [ ]:
515*515

In [ ]:
len(tcdf)

In [ ]:
ps = PoseSelector(name="default", variable="Pose_ID", number_to_return=1)

In [ ]:
one_pose = ps.run(posit_data)

In [ ]:
df2 = one_pose.dataframe
df2[(df2["Query_Ligand"] == "AAR-POS-0daf6b7e-1")&(df2["Reference_Ligand"] == "AAR-POS-d2a4d1df-21")]

In [ ]:
df["Pose_ID"].nunique()

In [ ]:
all(df.groupby(["Query_Ligand", "Reference_Ligand", "Type", "Aligned", "radius", "bitsize"]).count()) <=1

In [ ]:
df.groupby(["Query_Ligand", "Reference_Ligand", "Type", "Aligned", ]).count()

In [ ]:
all(tcdf.groupby(["Reference_Ligand", "Query_Ligand", "Type", "Aligned"]).count() <=1 )

In [ ]:
tcdf.groupby(["Reference_Ligand", "Query_Ligand", "Type", "Aligned"]).count() <=1

In [ ]:
import harbor.analysis.cross_docking as cd
from importlib import reload
reload(cd)

In [ ]:
tcdf[tcdf.groupby(["Reference_Ligand", "Query_Ligand", "Aligned"])[["Type"]].count() > 1]

In [ ]:
ri = tcdf.groupby(["Reference_Ligand", "Query_Ligand", "Aligned"])[["Type"]].count().reset_index()

In [ ]:
ri[ri["Type"] > 1]

In [ ]:
rows_with_values_greater_than_one = df[df.gt(1).any(axis=1)]

In [ ]:
newdf_model = cd.DataFrameModel(
            name="TanimotoComboData",
            type=cd.DataFrameType.CHEMICAL_SIMILARITY,
            dataframe=tcdf,
            key_columns=["Query_Ligand", "Reference_Ligand"],
            param_columns=["Aligned"]
        )

In [ ]:
grouped_by_key_and_param = newdf_model.dataframe.groupby(
            newdf_model.key_columns + newdf_model.param_columns
        ).count()
if len(grouped_by_key_and_param) == 0:
    raise KeyError(
        f"Grouping the dataframe by key_columns: '{newdf_model.key_columns}'"
        f"and param_columns: '{newdf_model.param_columns}"
        f"resulted in an empty dataframe"
    )
if not all(grouped_by_key_and_param <= 1):
    problem_rows = grouped_by_key_and_param[grouped_by_key_and_param > 1]
    raise KeyError(
        f"Grouping the dataframe by key_columns: '{newdf_model.key_columns}'"
        f"and param_columns: '{newdf_model.param_columns}"
        f"resulted in {len(problem_rows)} rows with duplicate values."
        f"Perhaps another column is needed!"
    )

In [ ]:
grouped_by_key_and_param[grouped_by_key_and_param["Tanimoto"] <= 1]

In [ ]:
(grouped_by_key_and_param > 1).any(axis=1).sum()

In [ ]:
grouped_by_key_and_param